In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]
dfs = [df.set_index("Date") for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

In [18]:
def convert_format(int_repr):
    s = str(int_repr)
    year = s[0:4]
    month = s[4:6]
    day = s[6:8]
    return year + "-" + month + "-" + day

na_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "North_America_5_Factors_Daily.csv")
na_factors_daily = na_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
na_factors_daily.index = [convert_format(x) for x in na_factors_daily.index]

europe_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "Europe_5_Factors_Daily.csv")
europe_factors_daily = europe_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
europe_factors_daily.index = [convert_format(x) for x in europe_factors_daily.index]

asia_pacific_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "Asia_Pacific_ex_Japan_5_Factors_Daily.csv")
asia_pacific_factors_daily = asia_pacific_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
asia_pacific_factors_daily.index = [convert_format(x) for x in asia_pacific_factors_daily.index]

In [19]:
# date intersections
def match_dataframes(compare_df, to_transform_df):
    truth = [x in compare_df.index for x in to_transform_df.index]
    # also add returns
    return to_transform_df[truth]

# need to transform factor format from old to etf format

new_snp = match_dataframes(na_factors_daily, dfs[0])
new_china = match_dataframes(asia_pacific_factors_daily, dfs[1])
new_em = match_dataframes(asia_pacific_factors_daily, dfs[2])
new_gold = match_dataframes(europe_factors_daily, dfs[3])
new_india = match_dataframes(europe_factors_daily, dfs[4])
new_mscieurope = match_dataframes(europe_factors_daily, dfs[5])
new_smallcapeurope = match_dataframes(europe_factors_daily, dfs[6])
dfs = [new_snp, new_china, new_em, new_gold, new_india, new_mscieurope, new_smallcapeurope]

na_factors_daily = match_dataframes(new_snp, na_factors_daily)
europe_factors_daily = match_dataframes(new_smallcapeurope, europe_factors_daily)
asia_pacific_factors_daily = match_dataframes(new_india, asia_pacific_factors_daily)

na_factors_daily = na_factors_daily.rename({old: old + "-NA" for old in na_factors_daily.columns}, axis=1)
europe_factors_daily = europe_factors_daily.rename({old: old + "-EU" for old in europe_factors_daily.columns}, axis=1)
asia_pacific_factors_daily = asia_pacific_factors_daily.rename({old: old + "-AS" for old in asia_pacific_factors_daily.columns}, axis=1)

factors_total = pd.concat([na_factors_daily, europe_factors_daily, asia_pacific_factors_daily], axis=1)
factors_total

,Mkt-RF-NA,SMB-NA,HML-NA,RMW-NA,CMA-NA,RF-NA,Mkt-RF-EU,SMB-EU,HML-EU,RMW-EU,CMA-EU,RF-EU,Mkt-RF-AS,SMB-AS,HML-AS,RMW-AS,CMA-AS,RF-AS
2015-04-14,0.17,-0.04,0.33,-0.03,0.13,0.00,0.39,0.39,0.06,0.13,-0.02,0.00,0.11,0.83,-0.30,-0.65,-0.40,0.00
2015-04-15,0.67,0.31,0.64,-0.26,-0.06,0.00,0.76,-0.08,0.41,-0.12,-0.03,0.00,-0.14,0.46,-0.06,-0.05,0.11,0.00
2015-04-16,-0.07,-0.05,-0.16,-0.15,-0.02,0.00,-0.05,0.35,-0.14,0.02,-0.10,0.00,1.11,0.72,-0.69,0.47,0.53,0.00
2015-04-17,-1.17,-0.35,0.23,0.08,0.09,0.00,-1.31,0.48,-0.21,0.03,-0.07,0.00,-0.57,0.69,0.35,-0.84,-0.18,0.00
2015-04-20,0.88,0.01,-0.16,0.40,-0.34,0.00,0.28,-0.34,0.17,-0.11,-0.05,0.00,-1.42,-0.18,0.59,-0.29,0.13,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,0.39,-0.67,0.01,0.65,0.10,0.01,0.48,-0.12,0.64,-0.16,0.30,0.01,0.28,0.02,0.58,-0.19,0.09,0.01
2026-01-27,0.37,-0.04,0.09,0.01,-0.45,0.01,1.89,-0.17,0.52,-0.02,0.05,0.01,1.90,-1.32,-0.17,0.73,-0.10,0.01
2026-01-28,-0.05,-0.25,0.31,0.02,-0.22,0.01,-1.46,0.73,0.86,-0.35,0.34,0.01,0.40,-0.23,0.84,0.01,-0.06,0.01
2026-01-29,-0.19,-0.13,1.45,0.87,0.97,0.01,-0.03,-0.24,0.57,0.66,0.44,0.01,-0.04,-0.60,1.03,0.22,0.32,0.01


Idea: for each ETF, run regression

$R_{ETF}-RF=\alpha+\beta_M(Mkt-RF)+\beta_S(SMB)+\beta_V(HML)+\epsilon$

In [20]:
def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

# construct returns matrix per asset
print(dfs[0]["Vol."])
T, N = dfs[0].shape
returns = np.empty((T,N))
volumes = np.empty((T,N))
prices = np.empty((T,N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    returns[:,i] = df["Price"].pct_change()
    returns[0,i] = 0.0
    volumes[:,i] = df["Vol."].apply(correct_volume).values
    prices[:,i] = df["Price"].values
    dfs_new.append(df)
dfs = dfs_new

print('returns:'); print(returns)

graph = go.Figure()
for i in range(returns.shape[1]):
    L = list(range(T))
    cum_returns = np.cumprod(1 + returns[:,i]) - 1
    graph.add_trace(go.Scatter(x=list(range(len(cum_returns))), y=cum_returns))
graph.show()

Date
2026-01-30    255.06K
2026-01-29    190.69K
2026-01-28    105.14K
2026-01-27    248.45K
2026-01-26    142.65K
               ...   
2015-04-20    136.50K
2015-04-17    151.06K
2015-04-16    122.76K
2015-04-15     59.67K
2015-04-14    120.74K
Name: Vol., Length: 2492, dtype: str
returns:
[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.00583242 -0.00188324 -0.00078462 ...  0.00178147  0.00061637
   0.00415324]
 [ 0.00041419  0.03962264  0.01177856 ... -0.02489627 -0.00517433
  -0.01030928]
 ...
 [-0.00126856 -0.00335008  0.00426223 ... -0.00271318 -0.01166407
   0.00143021]
 [-0.01054912  0.00672269 -0.00727567 ...  0.00388651 -0.00246525
  -0.00728363]
 [ 0.00437814 -0.01001669 -0.00793974 ...  0.0058072   0.00736145
   0.00258956]]


In [21]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):

    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]

        corr = np.corrcoef(section, rowvar=False) # (N,N)

        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w

        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    return all_corrs, all_volas


corrs, volas = compute_statistics_rolling(returns, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

corrs_mean = np.mean(corrs)
corrs_std = np.std(corrs)
volas_mean = np.mean(volas)
volas_std = np.std(volas)

mean corrs: 0.3811465880837967
mean volas: 0.008024740190772008
highest corrs: 0.6358583075346117
lowest corrs: 0.18405877676544133
lowest volas: 0.003906785081833062
highest volas: 0.02187760923699238


In [31]:
from scipy import odr

#import tensorflow as tf

alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.08
num_steps = 2000
beta = 0.1 # penalty volatility
min_weight = 1/11 # 1/N = 1/7, so a little less
tau = 0.5 # to start
a = 1.0 # for stress environment tuning
b = 1.0 # for stress environment tuning
factor_risk_exposure = 0.4

import tensorflow as tf

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x))
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def gradient(returns, factor_matrix, beta, volume):

    # returns, volume: (T, N)
    # factor matrix: (9, T)
    # beta: (N, 9)
    T, N = returns.shape
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    turnover_values = []
    X_tf = tf.convert_to_tensor(returns, dtype=tf.float32) # (T,N)

    corr = np.corrcoef(returns, rowvar=False)
    mean_corr = lower_part_mean(corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)
    w_prev_normalized = tf.nn.softmax(w_prev)

    factor_tf = tf.convert_to_tensor(factor_matrix, dtype=tf.float32)  # (9, T)
    beta_tf   = tf.convert_to_tensor(beta, dtype=tf.float32)          # (N, 9)
    volume_tf = tf.convert_to_tensor(volume, dtype=tf.float32)

    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_normalized = tf.reshape(w_normalized, [-1, 1])

            # risk model
            R_hat = tf.matmul(tf.transpose(factor_tf), tf.transpose(beta_tf)) # (T, N), predicted returns per factor
            portfolio_returns = (
                factor_risk_exposure * tf.matmul(R_hat, w_normalized)
                + (1 - factor_risk_exposure) * tf.matmul(X_tf, w_normalized)
            )
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(-portfolio_returns - t))
            
            # REMOVED FOR NOW:
            # portfolio_returns_from_factor = tf.matmul(R_hat, tf.reshape(w_col, [-1,1])) # (T,)
            # portfolio_returns_from_etfs = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T,)
            # portfolio_returns = factor_returns_weight * portfolio_returns_from_factor + (1 - factor_returns_weight) * portfolio_returns_from_etfs

            # build expected-return model
            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(volume_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(volume_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            residuals = X_tf - R_hat
            residual_var = tf.reduce_mean(tf.square(residuals), axis=0)  # (N,)

            idio_risk = tf.reduce_sum(residual_var * tf.square(tf.squeeze(w_normalized)))

            turnover_penalty = tau * tf.reduce_sum(tf.square(w_normalized - w_prev_normalized))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev_normalized)
            ).numpy()
            turnover_values.append(turnover)

            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_normalized))
            expected_return = tf.tensordot(mu, w_normalized, axes=1)
            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty + gamma*idio_risk

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev_normalized = tf.identity(w_normalized)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = returns @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    market_vol = tf.math.reduce_std(X_tf[-20:])
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = 1.0)
        return w_final / np.sum(w_final), True, stress_score, turnover_values
    return w_final, False, stress_score, turnover_values

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)

    cash_returns = np.zeros((validation_data.shape[0], 1))  # or use risk-free rate
    validation_data_with_cash = np.hstack([validation_data, cash_returns])

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    # perform metrics
    total_return = validation_data_with_cash @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w[:-1].numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md

def rolling_regression(returns, volumes, prices, total_factor, window_size, stepsize, validation_size):
    # all data is list of dataframes (concatenated)
    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    mds_w_opt = []
    mds_w_eqw = []

    stress_numbers = []
    turnover_all = []

    clip_counter = 0
    total_counter = 0

    weights_opt = []

    beta =  np.empty((N, 9)) # beta values
    for idx in range(0, T - window_size - validation_size, stepsize):
        beta_prev = beta.copy()
        total_factor_slice = total_factor.iloc[idx:idx+window_size]
        returns_slice = returns[idx:idx+window_size]
        volumes_slice = volumes[idx:idx+window_size]
        validation_returns = returns[idx+window_size:idx+window_size+validation_size]
        validation_prices = prices[idx+window_size:idx+window_size+validation_size]
        factor_matrix = np.array([
            total_factor_slice["Mkt-RF-NA"].values,
            total_factor_slice["Mkt-RF-EU"].values,
            total_factor_slice["Mkt-RF-AS"].values,
            total_factor_slice["SMB-NA"].values,
            total_factor_slice["SMB-EU"].values,
            total_factor_slice["SMB-AS"].values,
            total_factor_slice["HML-NA"].values,
            total_factor_slice["HML-EU"].values,
            total_factor_slice["HML-AS"].values
        ])
        for i in range(N):
            # for each asset: run regression independently
            output = odr.ODR(odr.Data(factor_matrix, returns_slice[:, i]), odr.multilinear).run()
            beta[i, :] = output.beta[1:] # skip constant term
    
        weighted_beta = 0.8 * beta_prev + 0.2 * beta # example weighting
        chosen_w, capped_weights, stress, mean_turnover = gradient(returns_slice, factor_matrix, weighted_beta, volumes_slice)
        weights_opt.append(chosen_w)
        if capped_weights:
            clip_counter += 1
        total_counter += 1
        turnover_all.append(mean_turnover)
        stress_numbers.append(stress)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(chosen_w, validation_returns, validation_prices)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)
        mds_w_opt.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(np.ones(N+1) / (N+1), validation_returns, validation_prices)
        # added one for cash

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)
        mds_w_eqw.append(md)

        df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
        "Highest maximum drawdowns": [np.min(mds_w_eqw), np.min(mds_w_opt)],
        "Mean maximum drawdowns": [np.mean(mds_w_eqw), np.mean(mds_w_opt)]
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, weights_opt, np.mean(weights_opt, axis=0), clip_counter / total_counter, stress_numbers, turnover_all

In [32]:
# run rolling window
df, weights_opt_all, weights_opt, clip_pct, stresses, turnover = rolling_regression(
    returns,
    volumes,
    prices,
    factors_total,
    window_size=252*2,
    stepsize=50,
    validation_size=252,
)

In [34]:
print('chosen weights (mean):')
print(weights_opt)
print()
print('chosen weights, first few:')
print(weights_opt_all[:5])
print()
print('chosen weights, last few:')
print(weights_opt_all[:-5])
print()
print('clip pct:')
print(clip_pct)
print()
print('stress values:')
print(stresses)
print()
print('turnovers:')
print(turnover)

chosen weights (mean):
[0.10762361 0.08517701 0.08980648 0.11960204 0.08518045 0.09029233
 0.18975514 0.23256291]

chosen weights, first few:
[[0.08515922 0.08515922 0.08515922 0.08515922 0.08515922 0.25028032
  0.08515922 0.23876438]
 [0.0874248  0.0874248  0.0874248  0.0874248  0.0874248  0.0874248
  0.23071909 0.24473219]
 [0.08497726 0.08497726 0.24703366 0.08497726 0.08497726 0.08497726
  0.08497726 0.24310273]
 [0.0733726  0.0733726  0.0733726  0.35299912 0.0733726  0.0733726
  0.0733726  0.20676534]
 [0.3701201  0.07128442 0.07128442 0.07128442 0.07128442 0.07128442
  0.07128442 0.2021734 ]]

chosen weights, last few:
[[0.08515922 0.08515922 0.08515922 0.08515922 0.08515922 0.25028032
  0.08515922 0.23876438]
 [0.0874248  0.0874248  0.0874248  0.0874248  0.0874248  0.0874248
  0.23071909 0.24473219]
 [0.08497726 0.08497726 0.24703366 0.08497726 0.08497726 0.08497726
  0.08497726 0.24310273]
 [0.0733726  0.0733726  0.0733726  0.35299912 0.0733726  0.0733726
  0.0733726  0.2067653

In [35]:
weights_graph = go.Figure()
L = list(range(len(weights_opt_all)))
for i in range(len(weights_opt_all[0])):
    to_plot = [w[i] for w in weights_opt_all]
    weights_graph.add_trace(go.Scatter(x=L, y=to_plot))
weights_graph.show()

In [36]:
df

,Sharpes,Sortinos,Vars,Cvars,MeanReturn,Highest maximum drawdowns,Mean maximum drawdowns
Equal weights,0.875541,1.205428,-0.007889,-0.013614,0.000355,-0.281273,-0.126898
Return + CVaR,0.905647,1.245794,-0.006809,-0.011720,0.000323,-0.292736,-0.127173


In [ ]:
"""
RESULTS

	Sharpes	Sortinos	Vars	Cvars	MeanReturn	Highest maximum drawdowns	Mean maximum drawdowns
Equal weights	0.875541	1.205428	-0.007889	-0.013614	0.000355	-0.051751	-0.126898
Return + CVaR	0.905240	1.245060	-0.006813	-0.011733	0.000323	-0.051804	-0.127191

chosen weights:
[0.10775131 0.08498918 0.08960699 0.11963961 0.08499047 0.08983943
 0.19124834 0.2319346 ]

clip pct:
0.7142857142857143

stress values:
[np.float64(0.739544186019144), np.float64(0.7867195288566188), np.float64(0.749714652840702), np.float64(0.48252446085636846), np.float64(0.4016125386197671), np.float64(0.37046113430865096), np.float64(0.39183191682935203), np.float64(0.5429447208803759), np.float64(0.3966976290526438), np.float64(0.5474095700187029), np.float64(0.5820992926292544), np.float64(0.9436248596803771), np.float64(0.9471695780771285), np.float64(0.9463751456127482), np.float64(0.9490783307753254), np.float64(0.9500027499179035), np.float64(0.9440017567630493), np.float64(0.9450718653533585), np.float64(0.9443224987936827), np.float64(0.9507819869936375), np.float64(0.9570887157922948), np.float64(0.7451265011261168), np.float64(0.7478523608935967), np.float64(0.7499685140721367), np.float64(0.7458106287035235), np.float64(0.7379062005469436), np.float64(0.7231235095314729), np.float64(0.7220173554052283), np.float64(0.728975146077121), np.float64(0.7043778225726062), np.float64(0.6257288129693455), np.float64(0.5541120000594162), np.float64(0.43205521634899247), np.float64(0.44650864410673335), np.float64(0.32540694234835266)]

turnovers:
[np.float32(0.0011137379), np.float32(0.001114457), np.float32(0.0011145285), np.float32(0.0011134007), np.float32(0.0011135539), np.float32(0.0011141265), np.float32(0.0011133688), np.float32(0.0011139167), np.float32(0.0011143229), np.float32(0.0011145327), np.float32(0.0011139567), np.float32(0.0011145233), np.float32(0.0011145428), np.float32(0.0011144984), np.float32(0.0011144496), np.float32(0.0011144076), np.float32(0.0011144077), np.float32(0.0011144269), np.float32(0.0011144), np.float32(0.001114433), np.float32(0.0011144422), np.float32(0.0011143918), np.float32(0.0011144166), np.float32(0.0011144571), np.float32(0.0011144874), np.float32(0.0011145177), np.float32(0.0011145291), np.float32(0.0011145107), np.float32(0.0011145241), np.float32(0.0011145185), np.float32(0.0011145361), np.float32(0.001114568), np.float32(0.0011145499), np.float32(0.0011145666), np.float32(0.0011145466)]

"""

In [37]:
def stress_test(df_data, total_factor, start_train, end_train, end_valid):
    
    returns_train = []
    prices_train = []
    volumes_train = []
    returns_val = []
    prices_val = []

    total_factor_copy = total_factor.copy()
    total_factor_copy.index = pd.to_datetime(total_factor_copy.index)

    total_factor_slice = total_factor_copy[
        (total_factor_copy.index >= pd.to_datetime(start_train))
        & (total_factor_copy.index <= pd.to_datetime(end_train))
    ]

    factor_matrix = np.array([
        total_factor_slice["Mkt-RF-NA"].values,
        total_factor_slice["Mkt-RF-EU"].values,
        total_factor_slice["Mkt-RF-AS"].values,
        total_factor_slice["SMB-NA"].values,
        total_factor_slice["SMB-EU"].values,
        total_factor_slice["SMB-AS"].values,
        total_factor_slice["HML-NA"].values,
        total_factor_slice["HML-EU"].values,
        total_factor_slice["HML-AS"].values
    ])

    beta =  np.empty((len(df_data), 9))

    for i,df in enumerate(df_data):
        df_datetime = df.copy()
        df_datetime.index = pd.to_datetime(df_datetime.index)
        train = df_datetime[
            (df_datetime.index >= pd.to_datetime(start_train))
            & (df_datetime.index <= pd.to_datetime(end_train))]
        val = df_datetime[
            (df_datetime.index >= pd.to_datetime(end_train))
            & (df_datetime.index <= pd.to_datetime(end_valid))
        ]

        l_temp = [0] + train["Price"].pct_change().values[1:].tolist()
        returns_train.append(l_temp)
        volumes_train.append(train["Vol."].apply(correct_volume).values)
        prices_train.append(train["Price"].values)
        prices_val.append(val["Price"].values)
        l_temp_val = [0] + val["Price"].pct_change().values[1:].tolist()
        returns_val.append(l_temp_val)
        # for each asset: run regression independently        
        output = odr.ODR(odr.Data(factor_matrix, l_temp), odr.multilinear).run()
        beta[i, :] = output.beta[1:] # skip constant term

    returns_train = np.array(returns_train).T
    volumes_train = np.array(volumes_train).T
    prices_train = np.array(prices_train).T
    returns_val = np.array(returns_val).T
    prices_val = np.array(prices_val).T

    print('shape returns_train:', returns_train.shape)
    print('shape factor_matrix:', factor_matrix.shape)
    print('beta shape:', beta.shape)
    print('voulmes train shape:', volumes_train.shape)
    
    w_opt, cut_weights, stress_score, turnover_values = gradient(returns_train, factor_matrix, beta, volumes_train)
    
    return w_opt, perform_validation(w_opt, returns_val, prices_val), perform_validation(np.ones(N+1) / (N+1), returns_val, prices_val), cut_weights, stress_score, turnover_values


In [38]:
start_train = "2019-10-01"
end_train = "2020-02-01"
end_valid = "2020-03-01"

print(factors_total)

chosen_weights, results_optimized, results_eqw, modes, stress, turnover_values = stress_test(
    dfs, factors_total, start_train, end_train, end_valid
)
var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()

            Mkt-RF-NA  SMB-NA  HML-NA  RMW-NA  CMA-NA  RF-NA  Mkt-RF-EU  \
2015-04-14       0.17   -0.04    0.33   -0.03    0.13   0.00       0.39   
2015-04-15       0.67    0.31    0.64   -0.26   -0.06   0.00       0.76   
2015-04-16      -0.07   -0.05   -0.16   -0.15   -0.02   0.00      -0.05   
2015-04-17      -1.17   -0.35    0.23    0.08    0.09   0.00      -1.31   
2015-04-20       0.88    0.01   -0.16    0.40   -0.34   0.00       0.28   
...               ...     ...     ...     ...     ...    ...        ...   
2026-01-26       0.39   -0.67    0.01    0.65    0.10   0.01       0.48   
2026-01-27       0.37   -0.04    0.09    0.01   -0.45   0.01       1.89   
2026-01-28      -0.05   -0.25    0.31    0.02   -0.22   0.01      -1.46   
2026-01-29      -0.19   -0.13    1.45    0.87    0.97   0.01      -0.03   
2026-01-30      -0.77   -1.14    0.07    0.54    0.76   0.01      -0.64   

            SMB-EU  HML-EU  RMW-EU  CMA-EU  RF-EU  Mkt-RF-AS  SMB-AS  HML-AS  \
2015-04-14    0.39 

In [ ]:
"""
RESULTS

OPTIMIZED WEIGHTS RESULTS
chosen weights: [0.32393646 0.07643975 0.07643975 0.07643975 0.07643975 0.07643975
 0.07643975 0.21742503]
var: -0.021787912
cvar: -0.026015347
sharpe: -4.041811837873361
sortino: -4.388310290265197
mean return: -0.0029829196
maximum drawdown: [-0.10985311]

EQUAL WEIGHTS RESULTS
var: -0.02177830718456484
cvar: -0.027318205932842975
sharpe: -2.849590648426675
sortino: -3.3882981375259766
mean return: -0.002250259582564444
maximum drawdown: [-0.10527719]

turnover values: 0.001113607
stress: 0.5585231203198049

"""

In [39]:
start_train = "2021-02-22"
end_train = "2022-01-01"
end_valid = "2023-01-23"

print(factors_total)

chosen_weights, results_optimized, results_eqw, modes, stress, turnover_values = stress_test(
    dfs, factors_total, start_train, end_train, end_valid
)
var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()

            Mkt-RF-NA  SMB-NA  HML-NA  RMW-NA  CMA-NA  RF-NA  Mkt-RF-EU  \
2015-04-14       0.17   -0.04    0.33   -0.03    0.13   0.00       0.39   
2015-04-15       0.67    0.31    0.64   -0.26   -0.06   0.00       0.76   
2015-04-16      -0.07   -0.05   -0.16   -0.15   -0.02   0.00      -0.05   
2015-04-17      -1.17   -0.35    0.23    0.08    0.09   0.00      -1.31   
2015-04-20       0.88    0.01   -0.16    0.40   -0.34   0.00       0.28   
...               ...     ...     ...     ...     ...    ...        ...   
2026-01-26       0.39   -0.67    0.01    0.65    0.10   0.01       0.48   
2026-01-27       0.37   -0.04    0.09    0.01   -0.45   0.01       1.89   
2026-01-28      -0.05   -0.25    0.31    0.02   -0.22   0.01      -1.46   
2026-01-29      -0.19   -0.13    1.45    0.87    0.97   0.01      -0.03   
2026-01-30      -0.77   -1.14    0.07    0.54    0.76   0.01      -0.64   

            SMB-EU  HML-EU  RMW-EU  CMA-EU  RF-EU  Mkt-RF-AS  SMB-AS  HML-AS  \
2015-04-14    0.39 